# Baseline Survival Analysis — Cox Proportional Hazards

This notebook builds the baseline survival analysis using the Cox proportional hazards model.  
We will:
1. Clean the dataset for CoxPH
2. Fit the Cox model and test proportional hazards
3. Stratify patients into risk groups
4. Visualize Kaplan–Meier curves by risk group
5. Document feature importance
6. Run Schoenfeld residual diagnostics


## Step 7: Clean Cox dataset
We transform and standardize variables for CoxPH:

- Encode categorical fields (age bands to ordinal, gender to binary, race to one‑hot with “Unknown” filled).  
- Standardize key numeric predictors to stabilize estimation.  
- Drop non‑predictive or high‑cardinality columns (diagnosis strings, medication flags, IDs, and target columns).  
- Confirm the final feature matrix contains only numeric columns plus the survival targets (`time`, `event`).  

**Key checks:**  
- Print dataset shape and total missing counts post‑cleaning.  
- Persist the scaler if you plan to score future data.

In [ ]:
# =============================================
# STEP 7 — CLEAN COX MODEL DATASET
# =============================================
df_cox = df.copy()

age_map = {
    '[10-20)': 1, '[20-30)': 2, '[30-40)': 3, '[40-50)': 4, '[50-60)': 5,
    '[60-70)': 6, '[70-80)': 7, '[80-90)': 8, '[90-100)': 9
}
df_cox["age"] = df_cox["age"].map(age_map).astype(float)
df_cox["gender"] = df_cox["gender"].map({"Male": 1, "Female": 0}).astype(float)

df_cox["race"] = df_cox["race"].fillna("Unknown")
df_cox = pd.get_dummies(df_cox, columns=["race"], drop_first=True)

# Standardize numeric predictors
scaler = StandardScaler()
numeric_cols = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses"
]
df_cox[numeric_cols] = scaler.fit_transform(df_cox[numeric_cols])

# Drop irrelevant columns
drop_cols = [
    "diag_1", "diag_2", "diag_3", "metformin", "repaglinide",
    "nateglinide", "chlorpropamide", "glimepiride", "acetohexamide",
    "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone",
    "tolazamide", "examide", "citoglipton", "insulin",
    "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone",
    "metformin-pioglitazone",
    "change", "diabetesMed", "payer_code", "medical_specialty",
    "readmitted", "readmitted_orig", "patient_nbr" 
]
df_cox = df_cox.drop(columns=[c for c in drop_cols if c in df_cox.columns])
df_cox = df_cox.select_dtypes(include=["float", "int"])

print("Cox dataset shape:", df_cox.shape)
print("Missing values:", df_cox.isna().sum().sum())

## Step 8: Fit Cox model and proportional hazards test
We fit a Cox proportional hazards model on the cleaned dataset:

- Use `time` as the duration column and `event` as the observed indicator.  
- Review the model summary: coefficients, hazard ratios, confidence intervals, and p‑values.  

Then evaluate the proportional hazards (PH) assumption:

- Run a global PH test to detect time‑dependent effects.  
- Note any variables violating PH and consider remedies (e.g., stratification, time interactions, or RSF as a non‑parametric alternative).  

**Deliverables:**  
- Save the fitted Cox model to disk for downstream use.  
- Record the concordance index from the fitted model as a baseline performance metric.

In [ ]:
# =============================================
# STEP 8 — FIT COX MODEL
# =============================================
cph = CoxPHFitter()
cph.fit(df_cox, duration_col="time", event_col="event")
cph.print_summary()

# Global PH test
results = proportional_hazard_test(cph, df_cox, time_transform="rank")
results.print_summary()

## Step 12: Risk stratification with quantiles
We translate continuous risk scores into interpretable groups:

- Compute individual risk using Cox partial hazards.  
- Split into quantile‑based tiers (Low, Medium, High risk via tertiles).  

**Why it matters:**  
- Clinicians and operations teams can target interventions for the High‑risk tier.  
- Quantile thresholds are simple to implement and communicate, yet can be tuned later.  

**Outputs:**  
- Counts per risk tier and a sample of patient‑level risk assignments.

# =============================================
# STEP 12 — RISK STRATIFICATION (Cox only)
# =============================================

# Compute Cox risk scores
df["cox_risk"] = cph.predict_partial_hazard(df_cox)

# Create 3 risk groups using quantiles
df["risk_group"] = pd.qcut(
    df["cox_risk"], 
    q=3, 
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

# Display group counts
print("\n=== RISK GROUP DISTRIBUTION ===")
print(df["risk_group"].value_counts())

# Show sample results
print("\n=== SAMPLE RISK STRATIFICATION ===")
print(df[["time", "event", "cox_risk", "risk_group"]].head())

## Step 13: Kaplan–Meier curves by risk group
We visualize survival differences across risk tiers:

- Plot KM curves for Low, Medium, and High risk groups on the same axis.  
- Run pairwise log‑rank tests to statistically confirm separation between groups.  

**Interpretation:**  
- Expect lower survival (higher readmission probability) in the High‑risk group.  
- Use these curves to justify thresholds for targeted follow‑up.  

**Artifacts:**  
- Save the KM plot by risk group to the `plots/` directory.

In [ ]:
# =============================================
# STEP 13 — KAPLAN–MEIER BY RISK GROUP
# =============================================

from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt
from lifelines.statistics import logrank_test
import os

os.makedirs("KM_Plots", exist_ok=True)

km = KaplanMeierFitter()

plt.figure(figsize=(8,5))

for group in ["Low Risk", "Medium Risk", "High Risk"]:
    mask = df["risk_group"] == group
    km.fit(
        durations=df.loc[mask, "time"],
        event_observed=df.loc[mask, "event"],
        label=group
    )
    km.plot()

plt.title("Kaplan–Meier Curves by Risk Group")
plt.xlabel("Days")
plt.ylabel("Survival Probability")
plt.savefig("KM_Plots/km_by_risk_group.png")
plt.show()


# Pairwise log-rank tests
low = df[df["risk_group"] == "Low Risk"]
med = df[df["risk_group"] == "Medium Risk"]
high = df[df["risk_group"] == "High Risk"]

print("Low vs Medium:")
print(logrank_test(low["time"], med["time"], low["event"], med["event"]).p_value)

print("Low vs High:")
print(logrank_test(low["time"], high["time"], low["event"], high["event"]).p_value)

print("Medium vs High:")
print(logrank_test(med["time"], high["time"], med["event"], high["event"]).p_value)

## Step 14: Cox feature importance
We document which variables drive the hazard:

- Rank features by absolute coefficient magnitude for effect size.  
- Report top features with coefficients, hazard ratios, and p‑values for significance.  

**Guidance:**  
- Positive coefficients increase hazard (higher readmission risk).  
- Focus interpretation on clinically actionable predictors (e.g., prior inpatient visits, medications, time in hospital).  

**Deliverables:**  
- Save a CSV of the full Cox summary table.  
- Export a bar chart of the top features to the `plots/` directory.

In [ ]:
# =============================================
# STEP 14 — FEATURE IMPORTANCE (COX MODEL — FINAL)
# =============================================

import os
import matplotlib.pyplot as plt

# Make sure folders exist
os.makedirs("EDA", exist_ok=True)
os.makedirs("report", exist_ok=True)

# Get Cox model summary
cox_summary = cph.summary.sort_values("p", ascending=True)

print("\n=== TOP 15 IMPORTANT FEATURES (COX MODEL) ===")
print(cox_summary[["coef", "exp(coef)", "p"]].head(15))

# Save full feature table
cox_summary.to_csv("report/cox_feature_importance.csv")

# Select top 15 by absolute effect size
top_feats = cox_summary.reindex(
    cox_summary["coef"].abs().sort_values(ascending=False).index
).head(15)

# Plot
plt.figure(figsize=(8,6))
top_feats["coef"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Top 15 Cox Model Feature Importances")
plt.xlabel("Coefficient (log hazard)")
plt.tight_layout()
plt.savefig("EDA/cox_feature_importance.png")
plt.show()

print("\n✅ Step 14 complete: Cox feature importance saved.")

## Step 15: Schoenfeld residual diagnostics and plots
We assess the PH assumption visually and numerically:

- Run Schoenfeld residual diagnostics for key variables.  
- Save diagnostic plots and the test results table.  

**What to look for:**  
- Systematic trends in residuals indicate time‑varying effects.  
- If violations are present, document them in limitations and consider model adjustments (stratification or non‑PH models).  

**Artifacts:**  
- CSV of PH test results.  
- A folder of diagnostic plots for reporting and audit.

In [ ]:
# =============================================
# STEP 15 — PROPORTIONAL HAZARDS CHECK
# =============================================

from lifelines.statistics import proportional_hazard_test
import matplotlib.pyplot as plt
import os

# Global PH test
results = proportional_hazard_test(cph, df_cox, time_transform='rank')

print("\n=== SCHOENFELD TEST RESULTS ===")
print(results.summary)

# ✅ ADD THIS LINE RIGHT HERE
results.summary.to_csv("report/schoenfeld_test_results.csv")

# Visual Schoenfeld plots
cph.check_assumptions(df_cox, p_value_threshold=0.05, show_plots=True)

# ✅ SAVE THE GENERATED PLOTS
os.makedirs("plots/PH_diagnostics", exist_ok=True)

for i in plt.get_fignums():
    fig = plt.figure(i)
    fig.savefig(f"plots/PH_diagnostics/schoenfeld_plot_{i}.png", dpi=300, bbox_inches="tight")

plt.close('all')